In [1]:
import math
import random
import os
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import spacy
from tqdm.auto import tqdm
import json

d:\Programming\nlp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
MAX_SEQ_LEN = 128
D_MODEL = 256
N_LAYERS = 4
N_HEADS = 8
DROPOUT = 0.1
BATCH_SIZE = 32
EPOCHS = 3
LR = 3e-4

In [4]:
MAX_VOCAB_SIZE = 300000
MIN_FREQ = 2

In [5]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
UNK_TOKEN = "<UNK>"
EOS_TOKEN = "<EOS>"
PAD_ID = 0
SOS_ID = 1
UNK_ID = 2
EOS_ID = 3

In [6]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len: int, embed_dim: int):
        super().__init__()
        self.embed_dim = embed_dim
        pe = torch.zeros(max_seq_len, self.embed_dim)
        for pos in range(max_seq_len):
            for i in range(0, self.embed_dim, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i) / self.embed_dim)))
                if i + 1 < self.embed_dim:
                    pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * (i + 1)) / self.embed_dim)))
        pe = pe.unsqueeze(0)  # (1, max_seq_len, embed_dim)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, embed_dim)
        seq_len = x.size(1)
        x = x * math.sqrt(self.embed_dim)
        x = x + self.pe[:, :seq_len, :].to(x.dtype)
        return x

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        """
        Standard multi-head attention:
          - q_proj/k_proj/v_proj: Linear(d_model, d_model)
          - split into heads: (batch, n_heads, seq, head_dim)
          - scaled dot-product per head
          - concat heads + out_proj
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # Learned linear projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, d_model) -> (batch, n_heads, seq, head_dim)
        b, seq, _ = x.size()
        return x.view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)

    def _combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, n_heads, seq, head_dim) -> (batch, seq, d_model)
        b, h, seq, hd = x.size()
        return x.transpose(1, 2).contiguous().view(b, seq, h * hd)

    def forward(self,
                x_q: torch.Tensor,
                x_kv: Optional[torch.Tensor] = None,
                attn_mask: Optional[torch.Tensor] = None,
                key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            x_q: (batch, seq_q, d_model)
            x_kv: (batch, seq_kv, d_model) or None -> self-attention
            attn_mask: causal mask or pairwise mask shape (seq_q, seq_kv) or (batch, seq_q, seq_kv) boolean (True = allowed)
            key_padding_mask: shape (batch, seq_kv) boolean where True means token is NOT padding (allowed).
        Returns:
            out: (batch, seq_q, d_model)
        """
        if x_kv is None:
            x_kv = x_q

        q = self.q_proj(x_q)  # (b, seq_q, d_model)
        k = self.k_proj(x_kv)  # (b, seq_kv, d_model)
        v = self.v_proj(x_kv)  # (b, seq_kv, d_model)

        q = self._split_heads(q)  # (b, h, seq_q, head_dim)
        k = self._split_heads(k)  # (b, h, seq_kv, head_dim)
        v = self._split_heads(v)  # (b, h, seq_kv, head_dim)

        # Scaled dot-product
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (b, h, seq_q, seq_kv)

        # Combine attn_mask and key_padding_mask into additive mask
        # attn_mask: boolean (True allowed). key_padding_mask: boolean (True = allowed token)
        if attn_mask is not None:
            # convert to additive (-inf where masked)
            if attn_mask.dtype == torch.bool:
                additive = (~attn_mask).to(scores.dtype) * -1e9
            else:
                additive = (1.0 - attn_mask).to(scores.dtype) * -1e9
            # broadcast to (batch, n_heads, seq_q, seq_kv)
            while additive.dim() < scores.dim():
                additive = additive.unsqueeze(0)
            scores = scores + additive

        if key_padding_mask is not None:
            # key_padding_mask: (batch, seq_kv) True means token allowed to attend
            # we want mask where False -> -inf
            kp = (~key_padding_mask).to(scores.dtype) * -1e9  # (batch, seq_kv)
            # expand to (batch, 1, 1, seq_kv) then broadcast
            kp = kp.unsqueeze(1).unsqueeze(1)  # (batch,1,1,seq_kv)
            scores = scores + kp

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)  # (b, h, seq_q, head_dim)
        out = self._combine_heads(out)  # (b, seq_q, d_model)
        out = self.out_proj(out)
        return out


In [8]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: Optional[int] = None, dropout: float = 0.1):
        super().__init__()
        if d_ff is None:
            d_ff = 4 * d_model
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None, key_padding_mask: Optional[torch.Tensor] = None):
        # masked self-attention
        sa = self.self_attn(x, x, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        x = x + self.dropout(sa)
        x = self.norm1(x)
        ff = self.ff(x)
        x = x + self.dropout(ff)
        x = self.norm2(x)
        return x

In [9]:
class DecoderOnlyModel(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, d_model: int = 256,
                 n_layers: int = 4, n_heads: int = 8, d_ff: Optional[int] = None,
                 dropout: float = 0.1, tie_weights: bool = True):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.n_heads = n_heads

        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_embedding = PositionalEmbedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff=d_ff, dropout=dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_weights:
            self.head.weight = self.token_embedding.weight

    def _causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        # boolean causal mask where True = allowed
        return torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool, device=device))

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None):
        """
        x: (batch, seq_len) token ids
        key_padding_mask: (batch, seq_len) boolean, True for non-pad tokens (allowed)
        returns logits: (batch, seq_len, vocab)
        """
        b, seq_len = x.size()
        assert seq_len <= self.max_seq_len, "input length exceeds model max_seq_len"

        tok = self.token_embedding(x)  # (b, seq_len, d_model)
        pos = self.pos_embedding(tok)  # (b, seq_len, d_model)
        h = self.dropout(pos)

        causal = self._causal_mask(seq_len, x.device)  # (seq, seq) boolean

        for block in self.blocks:
            h = block(h, attn_mask=causal, key_padding_mask=key_padding_mask)

        h = self.ln_f(h)
        logits = self.head(h)  # (b, seq_len, vocab)
        return logits

In [10]:
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm", disable=["ner"])

stopwords = nlp.Defaults.stop_words

In [11]:
def tokenize_and_normalize(doc):
    toks = []
    for tok in doc:
        if tok.is_space:
            continue
        if tok.is_punct or tok.is_digit:
            continue
        lemma = tok.lemma_.lower().strip()
        if lemma == "" or lemma in stopwords:
            continue
        toks.append(lemma)
    return toks

In [12]:
import multiprocessing
from collections import Counter

DATASET_NAME = "roneneldan/TinyStories"
TRAIN_SPLIT = "train"
VAL_SPLIT = "validation"
TOKENIZED_TRAIN_FNAME = "tokenized_train.jsonl"
TOKENIZED_VAL_FNAME = "tokenized_validation.jsonl"

In [13]:
def save_tokenized_list(tokenized_list, fname):
    with open(fname, "w", encoding="utf8") as f:
        for toks in tokenized_list:
            f.write(json.dumps(toks, ensure_ascii=False) + "\n")

def load_tokenized_list(fname):
    out = []
    with open(fname, "r", encoding="utf8") as f:
        for line in f:
            out.append(json.loads(line))
    return out

In [14]:
if os.path.exists(TOKENIZED_TRAIN_FNAME):
    print(f"Found {TOKENIZED_TRAIN_FNAME} — loading into memory.")
    tokenized_train = load_tokenized_list(TOKENIZED_TRAIN_FNAME)
    print("Loaded tokenized_train; examples:", len(tokenized_train))
else:
    print(f"{TOKENIZED_TRAIN_FNAME} not found. Downloading and tokenizing TRAIN split (one-time).")
    ds_train = load_dataset(DATASET_NAME, split="train")
    text_field = None
    for k in ds_train.column_names:
        if "text" in k.lower() or "story" in k.lower():
            text_field = k
            break
    if text_field is None:
        raise RuntimeError("Не найдено текстовое поле в TRAIN split.")
    train_texts = ds_train[text_field]

    n_process = max(1, min(4, multiprocessing.cpu_count() - 1))
    batch_size = 1000

    tokenized_train = []
    print(f"Tokenizing train with n_process={n_process}, batch_size={batch_size} ...")
    for doc in tqdm(nlp.pipe(train_texts, batch_size=batch_size, n_process=n_process),
                    total=len(train_texts), desc="Tokenizing train"):
        tokenized_train.append(tokenize_and_normalize(doc))

    save_tokenized_list(tokenized_train, TOKENIZED_TRAIN_FNAME)
    print("Saved", TOKENIZED_TRAIN_FNAME)

Found tokenized_train.jsonl — loading into memory.
Loaded tokenized_train; examples: 2119719


In [15]:
if os.path.exists(TOKENIZED_VAL_FNAME):
    print(f"Found {TOKENIZED_VAL_FNAME} — loading into memory.")
    tokenized_val = load_tokenized_list(TOKENIZED_VAL_FNAME)
    print("Loaded tokenized_val; examples:", len(tokenized_val))
else:
    print(f"{TOKENIZED_VAL_FNAME} not found. Downloading and tokenizing VALIDATION split (one-time).")
    ds_val = load_dataset(DATASET_NAME, split="validation")
    text_field = None
    for k in ds_val.column_names:
        if "text" in k.lower() or "story" in k.lower():
            text_field = k
            break
    if text_field is None:
        raise RuntimeError("Не найдено текстовое поле в VALIDATION split.")
    val_texts = ds_val[text_field]

    n_process = max(1, min(4, multiprocessing.cpu_count() - 1))
    batch_size = 1000

    tokenized_val = []
    print(f"Tokenizing validation with n_process={n_process}, batch_size={batch_size} ...")
    for doc in tqdm(nlp.pipe(val_texts, batch_size=batch_size, n_process=n_process),
                    total=len(val_texts), desc="Tokenizing val"):
        tokenized_val.append(tokenize_and_normalize(doc))

    save_tokenized_list(tokenized_val, TOKENIZED_VAL_FNAME)
    print("Saved", TOKENIZED_VAL_FNAME)

tokenized_validation.jsonl not found. Downloading and tokenizing VALIDATION split (one-time).


'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера' thrown while requesting HEAD https://huggingface.co/datasets/roneneldan/TinyStories/resolve/f54c09fd23315a6f9c86f9dc80f725de7d8f9c64/TinyStories.py
Retrying in 1s [Retry 1/5].
'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера' thrown while requesting HEAD https://huggingface.co/datasets/roneneldan/TinyStories/resolve/f54c09fd23315a6f9c86f9dc80f725de7d8f9c64/TinyStories.py
Retrying in 2s [Retry 2/5].
'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установ

Tokenizing validation with n_process=4, batch_size=1000 ...


Tokenizing val: 100%|██████████| 21990/21990 [01:43<00:00, 211.98it/s] 


Saved tokenized_validation.jsonl


In [16]:
print("Computing token frequencies from tokenized_train and tokenized_val ...")
freqs_train = Counter()
for toks in tqdm(tokenized_train, desc="Counting train"):
    freqs_train.update(toks)

freqs_val = Counter()
for toks in tqdm(tokenized_val, desc="Counting val"):
    freqs_val.update(toks)

print("Unique tokens in train freq:", len(freqs_train))
print("Unique tokens in val freq:", len(freqs_val))

print("Top 10 train tokens:", freqs_train.most_common(10))

Computing token frequencies from tokenized_train and tokenized_val ...


Counting val: 100%|██████████| 21990/21990 [00:00<00:00, 178915.88it/s]

Unique tokens in train freq: 60631
Unique tokens in val freq: 9379
Top 10 train tokens: [('lily', 3069249), ('day', 2715336), ('mom', 2329730), ('play', 2315304), ('want', 2041553), ('little', 1838135), ('time', 1828857), ('happy', 1751795), ('big', 1724717), ('look', 1632101)]


In [17]:
vocab_list = [PAD_TOKEN, SOS_TOKEN, UNK_TOKEN, EOS_TOKEN]

for tok, cnt in freqs_train.most_common():
    if cnt < MIN_FREQ:
        continue
    vocab_list.append(tok)
    if len(vocab_list) >= MAX_VOCAB_SIZE:
        break

token_to_id = {tok: idx for idx, tok in enumerate(vocab_list)}
id_to_token = {idx: tok for tok, idx in token_to_id.items()}

print("Vocab size:", len(vocab_list))

Vocab size: 41845


In [18]:
def encode(tokens: List[str]) -> List[int]:
    return [ token_to_id.get(t, UNK_ID) for t in tokens ]

In [19]:
def decode(ids: List[int]) -> str:
    toks = []
    for i in ids:
        if i == PAD_ID:
            continue
        if i == EOS_ID:
            break
        toks.append(id_to_token.get(i, UNK_TOKEN))
    return " ".join(toks)

In [20]:
def make_sequence(tokens: List[str], max_seq_len: int, token_to_id: dict):
    """
    tokens: список лемм/токенов истории (после предобработки)
    max_seq_len: желаемая длина финального id-листа (включая SOS и EOS)
    token_to_id: словарь
    Returns:
        ids: list[int] длины exactly max_seq_len
        start_len: int число токенов start (нужно для вычисления масок/оценок)
        end_len: int число токенов end
    """
    usable = max_seq_len - 2
    start_len = usable // 2
    end_len = usable - start_len

    start_tokens = tokens[:start_len]
    end_tokens = tokens[start_len:start_len + end_len]

    body_ids = [ token_to_id.get(t, UNK_ID) for t in (start_tokens + end_tokens) ]
    ids = [SOS_ID] + body_ids + [EOS_ID]

    if len(ids) < max_seq_len:
        ids = ids + [PAD_ID] * (max_seq_len - len(ids))
    elif len(ids) > max_seq_len:
        ids = ids[:max_seq_len]

    return ids, start_len, end_len

In [21]:
train_seqs = []
val_seqs = []

_example_seen = False
START_LEN = None
END_LEN = None

for toks in tqdm(tokenized_train, desc="Make train seqs"):
    ids, s_len, e_len = make_sequence(toks, MAX_SEQ_LEN, token_to_id)
    train_seqs.append(ids)
    if not _example_seen:
        START_LEN, END_LEN = s_len, e_len
        _example_seen = True

for toks in tqdm(tokenized_val, desc="Make val seqs"):
    ids, s_len, e_len = make_sequence(toks, MAX_SEQ_LEN, token_to_id)
    val_seqs.append(ids)

print(f"Created {len(train_seqs)} train sequences and {len(val_seqs)} val sequences.")
print(f"START_LEN={START_LEN}, END_LEN={END_LEN}")
print("Example sequence (ids):", train_seqs[0][:min(30, MAX_SEQ_LEN)])
print("Decoded example (tokens):", " ".join(id_to_token.get(i, '<UNK>') for i in train_seqs[0] if i not in (PAD_ID,)))

Make val seqs: 100%|██████████| 21990/21990 [00:00<00:00, 101974.13it/s]

Created 2119719 train sequences and 21990 val sequences.
START_LEN=63, END_LEN=63
Example sequence (ids): [1, 5, 9, 17, 4, 22, 1176, 80, 29, 974, 7, 605, 4, 8, 94, 1176, 6, 2191, 476, 714, 4, 6, 6, 22, 1176, 94, 2191, 714, 6, 18]
Decoded example (tokens): <SOS> day little girl lily find needle room know difficult play sharp lily want share needle mom sew button shirt lily mom mom find needle share sew shirt mom smile yes lily share needle fix shirt share needle sew button lily shirt difficult share help finish lily thank mom share needle fix shirt feel happy share work <EOS>


In [22]:
class StoryCausalDataset(Dataset):
    def __init__(self, sequences: List[List[int]]):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long)

def collate_pad(batch: List[torch.Tensor]):
    """
    batch: list of tensors shape (seq_len,) - in our pipeline seq_len == MAX_SEQ_LEN,
           but collate is robust to variable lengths.
    Returns:
        inputs: (batch, L)   where L = max length in batch - 1
        targets:(batch, L)
        key_padding_mask: (batch, L) boolean True for non-PAD tokens in inputs
    """
    lengths = [b.size(0) for b in batch]
    max_len = min(max(lengths), MAX_SEQ_LEN)
    padded = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    for i, b in enumerate(batch):
        L = min(b.size(0), max_len)
        padded[i, :L] = b[:L]
    # inputs / targets
    inputs = padded[:, :-1].contiguous()   # shape (batch, L-1)
    targets = padded[:, 1:].contiguous()   # shape (batch, L-1)
    key_padding_mask = (inputs != PAD_ID)  # True where real tokens (not pad)
    return inputs, targets, key_padding_mask

In [23]:
train_dataset = StoryCausalDataset(train_seqs)
val_dataset   = StoryCausalDataset(val_seqs)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pad)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pad)

In [24]:
model = DecoderOnlyModel(vocab_size=len(vocab_list), max_seq_len=MAX_SEQ_LEN,
                         d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
print("Model instantiated. d_model:", D_MODEL, "n_heads:", N_HEADS)

Model instantiated. d_model: 256 n_heads: 8


In [25]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for inputs, targets, key_padding_mask in tqdm(loader, desc="Train batches"):
        inputs = inputs.to(device)
        targets = targets.to(device)
        key_padding_mask = key_padding_mask.to(device)

        logits = model(inputs, key_padding_mask=key_padding_mask)  # (b, seq, vocab)
        logits_flat = logits.view(-1, logits.size(-1))
        targets_flat = targets.view(-1)
        loss = criterion(logits_flat, targets_flat)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

In [26]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets, key_padding_mask in tqdm(loader, desc="Eval batches"):
            inputs = inputs.to(device)
            targets = targets.to(device)
            key_padding_mask = key_padding_mask.to(device)
            logits = model(inputs, key_padding_mask=key_padding_mask)
            logits_flat = logits.view(-1, logits.size(-1))
            targets_flat = targets.view(-1)
            loss = criterion(logits_flat, targets_flat)
            total_loss += loss.item()
    return total_loss / len(loader)

In [27]:
best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    tr_loss = train_epoch(model, train_loader, optimizer, DEVICE)
    val_loss = evaluate(model, val_loader, DEVICE)
    print(f"Epoch {epoch} TrainLoss {tr_loss:.4f} ValLoss {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_decoder_multihead.pt")
        print("Saved best checkpoint.")


Eval batches: 100%|██████████| 688/688 [00:39<00:00, 17.25it/s]


Epoch 1 TrainLoss 4.6875 ValLoss 4.0103
Saved best checkpoint.


Train batches:  38%|███▊      | 24956/66242 [58:55<1:37:29,  7.06it/s]


KeyboardInterrupt: 

In [28]:
@torch.no_grad()
def generate(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 40, eos_id: Optional[int] = EOS_ID):
    model.eval()
    gen = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        if gen.size(1) > model.max_seq_len:
            gen = gen[:, -model.max_seq_len:]
        key_padding_mask = (gen != PAD_ID)
        logits = model(gen, key_padding_mask=key_padding_mask)  # (1, seq, vocab)
        next_logits = logits[:, -1, :]
        next_id = next_logits.argmax(dim=-1, keepdim=True)  # (1,1)
        gen = torch.cat([gen, next_id], dim=1)
        if eos_id is not None and next_id.item() == eos_id:
            break
    return gen.squeeze(0).tolist()

In [29]:
NUM_EXAMPLES = 5
for i in range(NUM_EXAMPLES):
    seq = val_seqs[i]
    # choose prompt = SOS + first half of core tokens
    core = seq[1:-1] if seq[-1] == EOS_ID else seq[1:]
    half = max(1, len(core)//2)
    prompt_ids = [SOS_ID] + core[:half]
    print(f"\nExample {i+1} prompt (decoded): {decode(prompt_ids)}")
    gen_ids = generate(model, prompt_ids, max_new_tokens=40)
    print("Generated:", decode(gen_ids))


Example 1 prompt (decoded): <SOS> spot spot shiny car wow kitty car bright clean kitty smile reply thank spot polish day play car kitty spot feel thirsty find small pond clear water drink water feel happy play day good friend
Generated: <SOS> spot spot shiny car wow kitty car bright clean kitty smile reply thank spot polish day play car kitty spot feel thirsty find small pond clear water drink water feel happy play day good friend

Example 2 prompt (decoded): <SOS> time big forest live rhinoceros roxy roxy love climb climb tree rock hill day roxy find icy hill like shiny cold want climb roxy try climb icy hill slippery try fall roxy sad want climb icy hill little bird billy billy roxy sad ask sad roxy roxy tell billy icy hill climb billy idea let find big leaf foot help climb icy hill
Generated: <SOS> time big forest live rhinoceros roxy roxy love climb climb tree rock hill day roxy find icy hill like shiny cold want climb roxy try climb icy hill slippery try fall roxy sad want climb 

In [35]:
def encode_text_to_ids(text: str) -> List[int]:
    try:
        nlp
    except NameError:
        import spacy
        nlp = spacy.load("en_core_web_sm", disable=["ner"])
    # produce spaCy doc and tokenize/normalize using existing function
    doc = nlp(text)
    toks = tokenize_and_normalize(doc)   # returns list[str]
    ids = [SOS_ID] + [ token_to_id.get(t, UNK_ID) for t in toks ]
    # ensure minimally two tokens so generation logic works
    if len(ids) < 2:
        ids = [SOS_ID, EOS_ID]
    return ids

In [ ]:
custom_prompts = [
    "Once upon a time there was a small cat",
    "A brave child found a magic stone",
    "The little robot wanted to"
]

print("Generating for custom prompts:")
for p in custom_prompts:
    pid = encode_text_to_ids(p)

    prompt_tokens = [ id_to_token.get(i, UNK_TOKEN) for i in pid ]
    print("\nPrompt:", p)
    print("Prompt tokens (ids):", pid)
    print("Prompt tokens (decoded):", prompt_tokens)

    gen_ids = generate(model, pid, max_new_tokens=50)

    if isinstance(gen_ids, torch.Tensor):
        gen_ids = gen_ids.tolist()

    decoded = []
    for idx in gen_ids:
        if idx == PAD_ID:
            continue
        if idx == EOS_ID:
            decoded.append("<EOS>")
            break
        decoded.append(id_to_token.get(idx, UNK_TOKEN))
    print("Generated (ids):", gen_ids)
    print("Generated (text):", " ".join(decoded))

Generating for custom prompts:

Prompt: Once upon a time there was a small cat
Prompt tokens (ids): [1, 10, 90, 102]
Prompt tokens (decoded): ['<SOS>', 'time', 'small', 'cat']
Generated (ids): [1, 10, 90, 102, 19, 19, 20, 7, 65, 131, 5, 19, 22, 12, 89, 488, 89, 19, 211, 8, 69, 89, 19, 79, 89, 22, 12, 89, 22, 12, 89, 19, 11, 19, 14, 9, 34, 180, 180, 180, 180, 180, 7, 89, 5, 92, 5, 92, 5, 92, 180, 19, 180, 7]
Generated (text): <SOS> time small cat tom tom love play outside sun day tom find big box yard box tom curious want inside box tom open box find big box find big box tom happy tom friend little bird sue sue sue sue sue play box day long day long day long sue tom sue play

Prompt: A brave child found a magic stone
Prompt tokens (ids): [1, 132, 203, 22, 489, 496]
Prompt tokens (decoded): ['<SOS>', 'brave', 'child', 'find', 'magic', 'stone']
Generated (ids): [1, 132, 203, 22, 489, 496, 8, 22, 68, 496, 13, 13, 22, 496, 13, 13, 22, 496, 40, 72, 59, 193, 25, 496, 193, 203, 13, 63, 53, 343